# Figure 2 Small: SLDS Reviewer Experiment

This notebook is intentionally separate from the figure 2 notebooks. Run the first section in the existing SMDS environment to export the simulated data, then switch to a separate environment with `lindermanlab/ssm` installed to fit the SLDS.

The default SLDS fit concatenates all trials into one long sequence with known trial boundaries. The model receives no trial identity, condition labels, block IDs, or drift coordinates. Trial boundaries are passed only so each trial is modeled as one LDS segment with one discrete state.


## 1. Export the simulated data in the SMDS environment

Run this section with the same environment used for `figure2_small.ipynb`. It reproduces the small simulated dataset and saves a compact `.npz` payload for the separate SLDS environment.


In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, '..')

import jax
jax.config.update('jax_enable_x64', True)

from jax import numpy as jnp
from jax import random as jr
import numpy as np

from dynamax.nonlinear_gaussian_ssm import StiefelManifoldSSM
from dynamax.utils.utils import random_rotation
from tensorflow_probability.substrates.jax.distributions import MultivariateNormalFullCovariance as MVN


In [ ]:
# Match the figure2_small simulation setup.
true_state_dim = 2
emission_dim = 10
num_trials = 750
num_conditions = 4
num_timesteps = 30
simulation_seed = 123
split_seed = 2626

dof = true_state_dim * (emission_dim - true_state_dim) + true_state_dim * (true_state_dim - 1) // 2

true_model = StiefelManifoldSSM(
    state_dim=true_state_dim,
    emission_dim=emission_dim,
    num_trials=num_trials,
    num_conditions=num_conditions,
)

key = jr.PRNGKey(simulation_seed)
dynamics = random_rotation(seed=key, n=true_state_dim, theta=jnp.pi / 5)

key, key_root = jr.split(key)
true_base_subspace = jr.orthogonal(key_root, emission_dim)

key, key_root = jr.split(key)
true_tau = jr.uniform(key_root, shape=(dof,), minval=1e-8, maxval=1e-4)

_velocity_cov = jnp.diag(true_tau)


def _get_velocity(prev_velocity, current_key):
    current_velocity_dist = MVN(loc=prev_velocity, covariance_matrix=_velocity_cov)
    current_velocity = current_velocity_dist.sample(seed=current_key)
    return current_velocity, current_velocity


keys = jr.split(key, num_trials)
key = keys[-1]
key, key_root = jr.split(key)
_initial_velocity = jnp.zeros(dof)
_, _velocity = jax.lax.scan(_get_velocity, _initial_velocity, keys[:-1])
_velocity = jnp.concatenate([_initial_velocity[None], _velocity])
true_tau = jnp.var(jnp.diff(_velocity, axis=0), axis=0)

key, key_root = jr.split(key)
true_params, param_props, true_velocity = true_model.initialize(
    tau=true_tau,
    base_subspace=true_base_subspace,
    key=key,
    initial_mean=jnp.sqrt(emission_dim / true_state_dim) * jr.normal(
        key_root, shape=(num_conditions, true_state_dim)
    ),
    dynamics_weights=dynamics,
    dynamics_covariance=jnp.eye(true_state_dim) * 1e-2,
    emission_covariance=jnp.eye(emission_dim) * 1e-2,
    velocity=_velocity,
)

conditions = jnp.tile(jnp.arange(num_conditions), num_trials)[:num_trials]
key, key_root = jr.split(key)
true_states, emissions, _ = true_model.sample(
    true_params, key, num_timesteps, conditions=conditions
)

print('emissions', emissions.shape)
print('true_states', true_states.shape)
print('true emission weights', true_params.emissions.weights.shape)


In [ ]:
# Match the figure2_small train/test metadata. This metadata is exported for diagnostics only.
block_size = 1
num_blocks = int(len(emissions) // block_size)
num_trials = num_blocks * block_size

emissions = emissions[:num_trials]
true_states = true_states[:num_trials]
conditions = conditions[:num_trials]

num_test_blocks = num_blocks // 6
split_key = jr.PRNGKey(split_seed)
test_idx = jax.random.choice(
    split_key,
    jnp.arange(30, num_blocks - 30, dtype=int),
    shape=(num_test_blocks,),
    replace=False,
)

block_masks = jnp.ones(num_blocks, dtype=bool).at[test_idx].set(False)
trial_masks = jnp.repeat(block_masks, block_size)

block_id_nums = jnp.repeat(
    jnp.arange(num_blocks, dtype=float) / (num_blocks - 1),
    block_size,
)
assert float(block_id_nums.min()) == 0.0
assert float(block_id_nums.max()) == 1.0

train_trial_idx = jnp.where(trial_masks)[0]
test_trial_idx = jnp.where(~trial_masks)[0]

print('num train trials', int(trial_masks.sum()))
print('num test trials', int((~trial_masks).sum()))
print('block_id_nums range', float(block_id_nums.min()), float(block_id_nums.max()))


In [ ]:
SLDS_OUT = Path('figure2_slds')
SLDS_OUT.mkdir(parents=True, exist_ok=True)
SLDS_DATA_PATH = SLDS_OUT / 'figure2_small_slds_data.npz'

np.savez_compressed(
    SLDS_DATA_PATH,
    emissions=np.asarray(emissions),
    true_states=np.asarray(true_states),
    true_emissions_weights=np.asarray(true_params.emissions.weights[:num_trials]),
    true_dynamics_weights=np.asarray(true_params.dynamics.weights),
    true_dynamics_covariance=np.asarray(true_params.dynamics.cov),
    true_emission_covariance=np.asarray(true_params.emissions.cov),
    true_initial_mean=np.asarray(true_params.initial.mean),
    true_tau=np.asarray(true_tau),
    true_velocity=np.asarray(true_velocity[:num_trials]),
    conditions=np.asarray(conditions),
    trial_idx=np.arange(num_trials),
    block_id_nums=np.asarray(block_id_nums),
    block_masks=np.asarray(block_masks),
    trial_masks=np.asarray(trial_masks),
    test_idx=np.asarray(test_idx),
    train_trial_idx=np.asarray(train_trial_idx),
    test_trial_idx=np.asarray(test_trial_idx),
    block_size=np.asarray(block_size),
    num_blocks=np.asarray(num_blocks),
    num_trials=np.asarray(num_trials),
    num_timesteps=np.asarray(num_timesteps),
    emission_dim=np.asarray(emission_dim),
    true_state_dim=np.asarray(true_state_dim),
    num_conditions=np.asarray(num_conditions),
    simulation_seed=np.asarray(simulation_seed),
    split_seed=np.asarray(split_seed),
)

print(f'Saved {SLDS_DATA_PATH}')
print('Run the remaining cells in the separate SLDS environment.')


## 2. Fit SLDS in the separate `ssm` environment

Switch kernels before running this section. The environment should install the trial-locked `jhdlee/ssm` branch plus NumPy, SciPy, and Matplotlib. This notebook does not install packages.


In [ ]:
from pathlib import Path
import json
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import subspace_angles

import ssm

SLDS_OUT = Path('figure2_slds')
SLDS_DATA_PATH = SLDS_OUT / 'figure2_small_slds_data.npz'

payload = np.load(SLDS_DATA_PATH, allow_pickle=False)

emissions = payload['emissions'].astype(float)
true_states = payload['true_states'].astype(float)
true_emissions_weights = payload['true_emissions_weights'].astype(float)
true_dynamics_weights = payload['true_dynamics_weights'].astype(float)
conditions = payload['conditions']
trial_idx = payload['trial_idx']
block_id_nums = payload['block_id_nums']
trial_masks = payload['trial_masks'].astype(bool)

num_trials, num_timesteps, emission_dim = emissions.shape
true_state_dim = int(payload['true_state_dim'])
num_conditions = int(payload['num_conditions'])

assert block_id_nums.min() == 0.0
assert block_id_nums.max() == 1.0

print('emissions', emissions.shape)
print('true_states', true_states.shape)
print('true_emissions_weights', true_emissions_weights.shape)
print('block_id_nums range', block_id_nums.min(), block_id_nums.max())


In [ ]:
# SLDS configuration.
# Trial boundaries are known; trial identity, conditions, block IDs, and drift coordinates are withheld.
FIT_MODE = 'trial_locked_concat'
TRANSITIONS = 'trial_locked'
DYNAMICS = 'trial_gaussian'
EMISSIONS = 'gaussian'
SINGLE_SUBSPACE = False

K_REQUESTED = 4  # 16
D = true_state_dim
K = K_REQUESTED

# Use ssm.initialize by default. K=4 keeps the PCA emissions initializer in
# the identifiable regime for this small experiment while the ARHMM initializer
# receives only trial boundaries, not trial identity or condition metadata.
USE_SSM_INITIALIZE = True
NUM_ITERS = 50
NUM_INIT_ITERS = 25
NUM_INIT_RESTARTS = 1
ALPHA = 0.0
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

fit_datas = [emissions.reshape(num_trials * num_timesteps, emission_dim)]
fit_tags = [{'trial_lengths': np.full(num_trials, num_timesteps, dtype=int)}]
fit_description = (
    'one concatenated sequence with known trial boundaries; no trial identity, '
    'conditions, block IDs, or drift coordinates'
)

INIT_TAG = 'ssm_init' if USE_SSM_INITIALIZE else 'random_init'
RUN_TAG = f'trial_locked_{EMISSIONS}_{INIT_TAG}_K{K}'

print(f'FIT_MODE={FIT_MODE}: {fit_description}')
print(f'TRANSITIONS={TRANSITIONS}, DYNAMICS={DYNAMICS}')
print(f'EMISSIONS={EMISSIONS}, SINGLE_SUBSPACE={SINGLE_SUBSPACE}')
print(f'K={K}, D={D}, N={emission_dim}, sequences={len(fit_datas)}')
print(f'USE_SSM_INITIALIZE={USE_SSM_INITIALIZE}, INIT_TAG={INIT_TAG}')


In [ ]:
# Fit a trial-locked SLDS with state-specific Gaussian emissions.
start_time = time.time()

slds = ssm.SLDS(
    emission_dim,
    K,
    D,
    transitions=TRANSITIONS,
    dynamics=DYNAMICS,
    emissions=EMISSIONS,
    single_subspace=SINGLE_SUBSPACE,
)

if USE_SSM_INITIALIZE:
    slds.initialize(
        fit_datas,
        tags=fit_tags,
        num_init_iters=NUM_INIT_ITERS,
        num_init_restarts=NUM_INIT_RESTARTS,
    )
else:
    print(
        'Skipping ssm.initialize, so no PCA emissions initialization or ARHMM '
        'initialization is used. Fitting starts from the SLDS constructor random parameters.'
    )

elbos, posterior = slds.fit(
    fit_datas,
    tags=fit_tags,
    method='laplace_em',
    variational_posterior='structured_meanfield',
    initialize=False,
    num_iters=NUM_ITERS,
    alpha=ALPHA,
)

elapsed_minutes = (time.time() - start_time) / 60.0
print(f'Finished SLDS fit in {elapsed_minutes:.2f} minutes')
print(f'Initial ELBO: {elbos[0]:.3f}')
print(f'Final ELBO: {elbos[-1]:.3f}')


In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 2.5))
ax.plot(elbos, color='black', lw=1.5)
ax.set_xlabel('iteration')
ax.set_ylabel('ELBO')
ax.set_title('SLDS fit')
fig.tight_layout()
fig.savefig(SLDS_OUT / f'slds_{RUN_TAG}_elbo.pdf')
plt.show()


## 3. Post hoc diagnostics

The next cells reshape the inferred states back into trial layout only after fitting. This checks whether the trial-locked SLDS discovered trial-aligned emission states without receiving trial labels, condition labels, block IDs, or drift coordinates during the fit.


In [ ]:
def _posterior_means_as_list(q):
    means = q.mean_continuous_states
    if isinstance(means, (list, tuple)):
        return list(means)
    return [means]


continuous_means = _posterior_means_as_list(posterior)
x_hat_concat = np.asarray(continuous_means[0])

z_hat_concat = np.asarray(
    slds.most_likely_trial_states(
        x_hat_concat,
        fit_datas[0],
        tag=fit_tags[0],
        expand=True,
    )
)
z_hat_trial = np.asarray(
    slds.most_likely_trial_states(
        x_hat_concat,
        fit_datas[0],
        tag=fit_tags[0],
        expand=False,
    )
)
trial_state_posterior, _ = slds.trial_state_expectations(
    x_hat_concat,
    fit_datas[0],
    tag=fit_tags[0],
)

z_hat_trials = z_hat_concat.reshape(num_trials, num_timesteps)
x_hat_trials = x_hat_concat.reshape(num_trials, num_timesteps, D)

state_counts = np.stack([
    np.bincount(z_hat_trials[i], minlength=K)
    for i in range(num_trials)
])
state_occupancy = state_counts / float(num_timesteps)
modal_state = z_hat_trial
modal_state_purity = state_counts.max(axis=1) / float(num_timesteps)
state_switches = (np.diff(z_hat_trials, axis=1) != 0).sum(axis=1)

assert np.all(state_switches == 0)

print('mean modal-state purity', modal_state_purity.mean())
print('median modal-state purity', np.median(modal_state_purity))
print('mean within-trial state switches', state_switches.mean())
print('states used', np.flatnonzero(state_counts.sum(axis=0)))


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 3.5))
im = ax.imshow(z_hat_trials, aspect='auto', interpolation='nearest', cmap='tab20')
ax.set_xlabel('time within trial')
ax.set_ylabel('trial')
ax.set_title('Inferred SLDS discrete state')
fig.colorbar(im, ax=ax, label='state')
fig.tight_layout()
fig.savefig(SLDS_OUT / f'slds_{RUN_TAG}_states_by_trial.pdf')
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(8.0, 5.0), sharex=True)
axes[0].plot(trial_idx, modal_state, lw=1.0, color='black')
axes[0].set_ylabel('modal state')
axes[1].plot(trial_idx, modal_state_purity, lw=1.0, color='tab:blue')
axes[1].set_ylabel('modal purity')
axes[1].set_ylim(0, 1.05)
axes[2].plot(trial_idx, state_switches, lw=1.0, color='tab:red')
axes[2].set_ylabel('switches')
axes[2].set_xlabel('trial')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(SLDS_OUT / f'slds_{RUN_TAG}_trial_state_summary.pdf')
plt.show()

fig, ax = plt.subplots(figsize=(8.0, 3.5))
im = ax.imshow(state_occupancy.T, aspect='auto', interpolation='nearest', cmap='viridis')
ax.set_xlabel('trial')
ax.set_ylabel('state')
ax.set_title('SLDS state occupancy by trial')
fig.colorbar(im, ax=ax, label='fraction of trial')
fig.tight_layout()
fig.savefig(SLDS_OUT / f'slds_{RUN_TAG}_state_occupancy.pdf')
plt.show()


In [ ]:
def grassmann_distance(A, B):
    """Normalized Grassmann distance between column spaces of A and B."""
    angles = subspace_angles(A, B)
    return np.linalg.norm(angles) / (np.sqrt(A.shape[1]) * (np.pi / 2.0))


learned_emissions = np.asarray(slds.emissions.Cs)
if learned_emissions.shape[0] != K:
    raise RuntimeError(
        f'Expected K={K} learned emission matrices; got shape {learned_emissions.shape}. '
        'Check that single_subspace=False was honored by the installed ssm version.'
    )

emission_distance = np.empty((num_trials, K))
for t in range(num_trials):
    for k in range(K):
        emission_distance[t, k] = grassmann_distance(
            true_emissions_weights[t],
            learned_emissions[k],
        )

modal_emission_distance = emission_distance[np.arange(num_trials), modal_state]
nearest_state_by_emission = emission_distance.argmin(axis=1)
nearest_emission_distance = emission_distance.min(axis=1)

print('mean modal-state emission distance', modal_emission_distance.mean())
print('mean nearest-state emission distance', nearest_emission_distance.mean())


In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 3.0))
ax.plot(trial_idx, modal_emission_distance, lw=1.0, label='modal inferred state')
ax.plot(trial_idx, nearest_emission_distance, lw=1.0, label='nearest inferred emission state')
ax.set_xlabel('trial')
ax.set_ylabel('normalized subspace distance')
ax.set_title('True trial emission subspace vs learned SLDS emissions')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(SLDS_OUT / f'slds_{RUN_TAG}_emission_subspace_distances.pdf')
plt.show()

fig, ax = plt.subplots(figsize=(8.0, 3.5))
im = ax.imshow(emission_distance.T, aspect='auto', interpolation='nearest', cmap='magma_r')
ax.set_xlabel('trial')
ax.set_ylabel('SLDS state')
ax.set_title('Subspace distance to each learned emission state')
fig.colorbar(im, ax=ax, label='normalized distance')
fig.tight_layout()
fig.savefig(SLDS_OUT / f'slds_{RUN_TAG}_emission_distance_heatmap.pdf')
plt.show()


In [ ]:
learned_dynamics = np.asarray(slds.dynamics.As)
if learned_dynamics.shape[0] != K:
    raise RuntimeError(f'Expected K={K} dynamics matrices; got shape {learned_dynamics.shape}')

dynamics_distance = np.empty((K, K))
for i in range(K):
    for j in range(K):
        dynamics_distance[i, j] = np.linalg.norm(learned_dynamics[i] - learned_dynamics[j], ord='fro')

learned_eigs = np.linalg.eigvals(learned_dynamics)
true_eigs = np.linalg.eigvals(true_dynamics_weights)

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
im = axes[0].imshow(dynamics_distance, cmap='viridis')
axes[0].set_xlabel('state')
axes[0].set_ylabel('state')
axes[0].set_title('Pairwise dynamics distance')
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

unit = np.exp(1j * np.linspace(0, 2 * np.pi, 300))
axes[1].plot(unit.real, unit.imag, color='0.7', lw=1.0)
axes[1].scatter(true_eigs.real, true_eigs.imag, color='black', s=45, label='true')
for k in range(K):
    axes[1].scatter(learned_eigs[k].real, learned_eigs[k].imag, s=25, alpha=0.8)
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_xlabel('real')
axes[1].set_ylabel('imag')
axes[1].set_title('Learned SLDS eigenvalues')
axes[1].legend(frameon=False, fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(SLDS_OUT / f'slds_{RUN_TAG}_dynamics_diagnostics.pdf')
plt.show()


In [ ]:
summary_path = SLDS_OUT / f'slds_{RUN_TAG}_summary.npz'
np.savez_compressed(
    summary_path,
    z_hat_trials=z_hat_trials,
    x_hat_trials=x_hat_trials,
    state_counts=state_counts,
    state_occupancy=state_occupancy,
    modal_state=modal_state,
    modal_state_purity=modal_state_purity,
    state_switches=state_switches,
    learned_emissions=learned_emissions,
    emission_distance=emission_distance,
    modal_emission_distance=modal_emission_distance,
    nearest_state_by_emission=nearest_state_by_emission,
    nearest_emission_distance=nearest_emission_distance,
    learned_dynamics=learned_dynamics,
    dynamics_distance=dynamics_distance,
    learned_eigs=learned_eigs,
    true_eigs=true_eigs,
    elbos=np.asarray(elbos),
    K=np.asarray(K),
    K_requested=np.asarray(K_REQUESTED),
    use_ssm_initialize=np.asarray(USE_SSM_INITIALIZE),
    transitions=np.asarray(TRANSITIONS),
    dynamics=np.asarray(DYNAMICS),
    trial_lengths=np.asarray(fit_tags[0]['trial_lengths']),
    trial_state_posterior=trial_state_posterior,
    init_tag=np.asarray(INIT_TAG),
    emissions=np.asarray(EMISSIONS),
    single_subspace=np.asarray(SINGLE_SUBSPACE),
    run_tag=np.asarray(RUN_TAG),
    D=np.asarray(D),
    fit_mode=np.asarray(FIT_MODE),
    random_seed=np.asarray(RANDOM_SEED),
)
print(f'Saved {summary_path}')

pickle_path = SLDS_OUT / f'slds_{RUN_TAG}_fit.pkl'
try:
    with open(pickle_path, 'wb') as f:
        pickle.dump(
            {
                'model': slds,
                'posterior': posterior,
                'elbos': elbos,
                'config': {
                    'fit_mode': FIT_MODE,
                    'K': K,
                    'K_requested': K_REQUESTED,
                    'use_ssm_initialize': USE_SSM_INITIALIZE,
    'transitions': TRANSITIONS,
    'dynamics': DYNAMICS,
                    'transitions': TRANSITIONS,
                    'dynamics': DYNAMICS,
                    'init_tag': INIT_TAG,
                    'emissions': EMISSIONS,
                    'single_subspace': SINGLE_SUBSPACE,
                    'run_tag': RUN_TAG,
                    'D': D,
                    'num_iters': NUM_ITERS,
                    'num_init_iters': NUM_INIT_ITERS,
                    'num_init_restarts': NUM_INIT_RESTARTS,
                    'alpha': ALPHA,
                    'random_seed': RANDOM_SEED,
                },
            },
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    print(f'Saved {pickle_path}')
except Exception as exc:
    print(f'Could not pickle full SLDS fit: {exc}')

manifest_path = SLDS_OUT / f'slds_{RUN_TAG}_manifest.json'
manifest = {
    'data_path': str(SLDS_DATA_PATH),
    'summary_path': str(summary_path),
    'pickle_path': str(pickle_path),
    'fit_description': fit_description,
    'fit_mode': FIT_MODE,
    'K': K,
    'K_requested': K_REQUESTED,
    'use_ssm_initialize': USE_SSM_INITIALIZE,
    'transitions': TRANSITIONS,
    'dynamics': DYNAMICS,
    'init_tag': INIT_TAG,
    'emissions': EMISSIONS,
    'single_subspace': SINGLE_SUBSPACE,
    'run_tag': RUN_TAG,
    'D': D,
    'num_iters': NUM_ITERS,
    'num_init_iters': NUM_INIT_ITERS,
    'num_init_restarts': NUM_INIT_RESTARTS,
    'alpha': ALPHA,
    'random_seed': RANDOM_SEED,
    'num_trials': int(num_trials),
    'num_timesteps': int(num_timesteps),
    'emission_dim': int(emission_dim),
}
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Saved {manifest_path}')


## Interpretation guide

If SLDS explains the drift by using trial-aligned emission states, the inferred state image should show mostly horizontal bands across time within each trial, per-trial modal-state purity should be high, and the modal state's emission subspace should be close to that trial's true emission subspace. If the inferred states primarily switch within trial or the modal-state emission distances stay high, that argues against SLDS recovering the trial-specific emission drift in this no-trial-information setting.
